# Evaluation Accuracy Calculation

This notebook calculates the accuracy of the baseline model and the RAG model using manually labeled evaluation results.

In [1]:
import pandas as pd

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content/drive/MyDrive

/content/drive/MyDrive


In [11]:
# Load CSV
df = pd.read_csv("evaluation_results_fill.csv")
df.head()

,question_id,question,baseline_answer,rag_answer,top_text,top_similarity,top_lecture,top_page,retrieved_context,retrieved_sources,retrieved_contains_answer,rag_correct,baseline_correct,notes
0,1,What is the bottleneck problem in encoder-decoder models?,"The bottleneck problem in encoder-decoder models, particularly in the contex...",The bottleneck problem in encoder-decoder models refers to the challenge of ...,From Lecture 5:\nInput sequence -> Encoder -> Fixed-size context vector -> D...,0.596855,Lecture 6 - Attention Mechanisms,2,"[Source: Lecture 6 - Attention Mechanisms, Page 2]\nFrom Lecture 5:\nInput s...",Lecture 6 - Attention Mechanisms p.2 sim=0.597 | Lecture 9 - Pre-training LL...,Yes,Yes,Partial,Dataset clearly contains the bottleneck definition. RAG answer is concise an...
1,2,Why do fixed-size context vectors lose information?,"Fixed-size context vectors, often used in traditional models like RNNs and e...",Fixed-size context vectors lose information because they attempt to compress...,What we learned today\nAttention solves the bottleneck problem - dynamic con...,0.546057,Lecture 6 - Attention Mechanisms,18,"[Source: Lecture 6 - Attention Mechanisms, Page 18]\nWhat we learned today\n...",Lecture 6 - Attention Mechanisms p.18 sim=0.546 | Lecture 6 - Attention Mech...,Yes,Yes,Partial,Dataset clearly explains that compressing long sequences into a fixed-size v...
2,3,How does attention solve the bottleneck problem?,"The bottleneck problem in sequence processing tasks, such as machine transla...",Attention solves the bottleneck problem by using a dynamic context instead o...,What we learned today\nAttention solves the bottleneck problem - dynamic con...,0.647113,Lecture 6 - Attention Mechanisms,18,"[Source: Lecture 6 - Attention Mechanisms, Page 18]\nWhat we learned today\n...",Lecture 6 - Attention Mechanisms p.18 sim=0.647 | Lecture 6 - Attention Mech...,Yes,Yes,Partial,RAG directly explains dynamic context solving bottleneck. Baseline is correc...
3,4,What is the intuition behind attention?,"The intuition behind attention in the context of machine learning, particula...",The intuition behind attention is to allow a model to focus on different par...,This is exactly how attention works!\nAttention beyond translation\nTranslat...,0.624338,Lecture 6 - Attention Mechanisms,4,"[Source: Lecture 6 - Attention Mechanisms, Page 4]\nThis is exactly how atte...",Lecture 6 - Attention Mechanisms p.4 sim=0.624 | Lecture 6 - Attention Mecha...,Yes,Yes,Partial,RAG captures core intuition clearly. Baseline is verbose and includes broade...
4,5,"What are Query, Key, and Value in attention?","In the context of attention mechanisms, particularly in transformer models, ...","In attention mechanisms, Query (Q), Key (K), and Value (V) are three differe...",Three roles in attention\nAttention uses three diﬀerent representations of t...,0.705792,Lecture 6 - Attention Mechanisms,3,"[Source: Lecture 6 - Attention Mechanisms, Page 3]\nThree roles in attention...",Lecture 6 - Attention Mechanisms p.3 sim=0.706 | Lecture 6 - Attention Mecha...,Yes,Yes,Partial,RAG aligns with lecture Q/K/V roles. Baseline is correct but overly detailed...


In [12]:
# Check columns
print(df.columns)

Index(['question_id', 'question', 'baseline_answer', 'rag_answer', 'top_text',
       'top_similarity', 'top_lecture', 'top_page', 'retrieved_context',
       'retrieved_sources', 'retrieved_contains_answer', 'rag_correct',
       'baseline_correct', 'notes'],
      dtype='object')


## Convert manual labels to scores

This step converts the manual evaluation labels into numerical scores:

*   Yes → 1
*   Partial → 0.5
*   No → 0

These scores are used to compute the overall accuracy.

In [13]:
score_map = {"Yes": 1, "Partial": 0.5, "No": 0}

df["rag_score"] = df["rag_correct"].map(score_map)
df["baseline_score"] = df["baseline_correct"].map(score_map)

df[["question_id", "rag_correct", "baseline_correct", "rag_score", "baseline_score"]].head()

,question_id,rag_correct,baseline_correct,rag_score,baseline_score
0,1,Yes,Partial,1.0,0.5
1,2,Yes,Partial,1.0,0.5
2,3,Yes,Partial,1.0,0.5
3,4,Yes,Partial,1.0,0.5
4,5,Yes,Partial,1.0,0.5


## Calculate accuracy

This step prints the final accuracy results for both the baseline and RAG models based on manual evaluation.

In [16]:
rag_accuracy = df["rag_score"].mean()
baseline_accuracy = df["baseline_score"].mean()

print(f"RAG Accuracy: {rag_accuracy:.2%}")
print(f"Baseline Accuracy: {baseline_accuracy:.2%}")

RAG Accuracy: 98.00%
Baseline Accuracy: 53.00%


In [17]:
# Strict RAG accuracy
df["rag_score_strict"] = df["rag_correct"].apply(lambda x: 1 if x == "Yes" else 0)
rag_accuracy_strict = df["rag_score_strict"].mean()

print(f"Strict RAG Accuracy: {rag_accuracy_strict:.2%}")

Strict RAG Accuracy: 96.00%


In [19]:
# Average top_similarity
avg_similarity = df["top_similarity"].mean()

print(f"Average Similarity: {avg_similarity:.2f}")

Average Similarity: 0.60


## Conclusion

We evaluate the performance of the baseline and RAG models using manual accuracy and semantic similarity. The accuracy is computed based on human-labeled results, which may introduce some bias.

The RAG model achieves a soft accuracy of 98% and a strict accuracy of 96%, while the baseline model only achieves 53%. This shows that the RAG model performs significantly better than the baseline. The manual evaluation is based on whether the RAG answers are aligned with the lecture materials.

The high accuracy of the RAG model is mainly due to the fact that the questions are directly derived from lecture content, and the retrieved chunks often contain the relevant information. Therefore, the model is able to generate answers that are consistent with the lecture materials.

We also compute the average semantic similarity, which is 0.60. This indicates that although the answers are generally correct in meaning, they may differ in wording.

It is important to note that manual evaluation may introduce subjectivity, and some answers marked as correct may not fully cover all necessary details. In addition, the dataset is limited to lecture materials, which may make the task easier for a retrieval-based system.

## Error Analysis

Although the RAG model achieves high accuracy, it still fails in some cases.

One common issue is retrieval errors. In some cases, the retrieved chunks do not contain the most relevant information, or the retrieved content is not accurate enough. This can lead to incorrect or incomplete answers.

Another issue is that some answers are too general and do not fully address the question, even though they may be partially correct.

In addition, some questions are not clearly aligned with the lecture content, which can make it difficult for the retrieval process to find the most relevant information, affecting the quality of the RAG responses.

For the baseline model, since there is no retrieval mechanism, the generated answers are often very long and may include inaccurate or irrelevant information. These responses tend to lack focus and include unnecessary details.

These errors indicate that the performance of the RAG system still depends heavily on the quality of retrieval and the completeness of generated responses. Due to limited time, further optimization of the RAG system has not been explored, but future improvements could focus on enhancing retrieval strategies and answer generation.